# Mesh Tutorial 5: Quality, Validation, and Repair

This tutorial covers mesh quality assessment and repair:

1. **Quality Metrics**: Aspect ratio, angles, edge lengths
2. **Mesh Statistics**: Summary of mesh properties
3. **Validation**: Detect common mesh errors
4. **Repair Operations**: Fix mesh problems
5. **Topology Checks**: Watertight and manifold validation

---

## Why Mesh Quality Matters

Poor mesh quality can cause:
- **Numerical instability** in PDE solvers
- **Inaccurate physics** from distorted elements
- **Training issues** for ML models (garbage in, garbage out)
- **Visualization artifacts** from degenerate geometry

In [ ]:
import torch

from physicsnemo.mesh import Mesh
from physicsnemo.mesh.primitives.surfaces import sphere_icosahedral
from physicsnemo.mesh.primitives.planar import unit_square
from physicsnemo.mesh.primitives.procedural import lumpy_sphere

## Section 1: Quality Metrics

PhysicsNeMo-Mesh computes per-cell quality metrics that help identify problematic elements.

In [ ]:
# Load a mesh
mesh = sphere_icosahedral.load(subdivisions=2)

# Get quality metrics
metrics = mesh.quality_metrics

print("Quality metrics (per cell):")
for key in metrics.keys():
    values = metrics[key]
    if not values.isnan().all():
        print(f"  {key}: min={values.min():.4f}, max={values.max():.4f}, mean={values.mean():.4f}")

### Understanding Quality Metrics

| Metric | Ideal Value | Meaning |
|--------|------------|----------|
| `aspect_ratio` | 1.0 | Ratio of max edge to characteristic length |
| `edge_length_ratio` | 1.0 | Ratio of max to min edge length |
| `min_angle` | π/3 (60°) | Smallest interior angle |
| `max_angle` | π/3 (60°) | Largest interior angle |
| `quality_score` | 1.0 | Combined metric (1.0 = perfect equilateral) |

In [ ]:
import math

# Visualize quality score
mesh.cell_data["quality"] = metrics["quality_score"]

print(f"Quality score range: [{metrics['quality_score'].min():.3f}, {metrics['quality_score'].max():.3f}]")
print(f"Ideal equilateral triangle: 1.0")

mesh.draw(cell_scalars="quality", cmap="RdYlGn", show_edges=True)

In [ ]:
# Identify poor quality cells
quality_threshold = 0.5
poor_quality_mask = metrics["quality_score"] < quality_threshold
n_poor = poor_quality_mask.sum().item()

print(f"Cells with quality < {quality_threshold}: {n_poor} / {mesh.n_cells} ({100*n_poor/mesh.n_cells:.1f}%)")

### Comparing Mesh Quality

Let's compare quality between different mesh types.

In [ ]:
# Regular mesh (high quality)
regular = sphere_icosahedral.load(subdivisions=3)
regular_quality = regular.quality_metrics["quality_score"].mean()

# Perturbed mesh (lower quality)
lumpy = lumpy_sphere.load(noise_amplitude=0.3, subdivisions=3, seed=42)
lumpy_quality = lumpy.quality_metrics["quality_score"].mean()

print(f"Regular sphere mean quality: {regular_quality:.4f}")
print(f"Lumpy sphere mean quality: {lumpy_quality:.4f}")

## Section 2: Mesh Statistics

Get a comprehensive summary of mesh properties.

In [ ]:
mesh = torch.load("assets/bunny.pt", weights_only=False).subdivide(1, "loop")

stats = mesh.statistics

print("Mesh Statistics:")
print("=" * 40)
for key, value in stats.items():
    if isinstance(value, (int, float)):
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")
    elif isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            if isinstance(v, float):
                print(f"    {k}: {v:.4f}")
            else:
                print(f"    {k}: {v}")

## Section 3: Mesh Validation

The `validate()` method checks for common mesh errors.

In [ ]:
# Validate a good mesh
mesh = sphere_icosahedral.load(subdivisions=2)
report = mesh.validate()

print("Validation Report (good mesh):")
print(f"  Valid: {report['valid']}")
if report.get('errors'):
    print(f"  Errors: {report['errors']}")
if report.get('warnings'):
    print(f"  Warnings: {report['warnings']}")

In [ ]:
# Create a mesh with some problems
points = torch.tensor([
    [0.0, 0.0],
    [1.0, 0.0],
    [0.5, 1.0],
    [0.5, 0.5],  # Interior point (will be unused)
    [0.0, 0.0],  # Duplicate of point 0
])

cells = torch.tensor([
    [0, 1, 2],  # Valid triangle
    [0, 0, 1],  # Degenerate (repeated vertex)
])

bad_mesh = Mesh(points=points, cells=cells)

# Validate
report = bad_mesh.validate(
    check_degenerate_cells=True,
    check_duplicate_vertices=True,
)

print("Validation Report (bad mesh):")
print(f"  Valid: {report['valid']}")
for key, value in report.items():
    if key not in ['valid'] and value:
        print(f"  {key}: {value}")

## Section 4: Repair Operations

PhysicsNeMo-Mesh provides several repair operations.

### All-in-One: mesh.clean()

In [ ]:
# Create a mesh with duplicate points
points = torch.tensor([
    [0.0, 0.0],
    [1.0, 0.0],
    [0.5, 1.0],
    [0.0, 0.0],  # Duplicate of point 0
    [1.0, 0.0],  # Duplicate of point 1
])

cells = torch.tensor([
    [0, 1, 2],  # Triangle using original points
    [3, 4, 2],  # Triangle using duplicate points
])

mesh_with_duplicates = Mesh(points=points, cells=cells)
print(f"Before cleaning: {mesh_with_duplicates.n_points} points, {mesh_with_duplicates.n_cells} cells")

# Clean the mesh
cleaned = mesh_with_duplicates.clean()
print(f"After cleaning: {cleaned.n_points} points, {cleaned.n_cells} cells")

### Detailed Repair Pipeline

In [ ]:
from physicsnemo.mesh.repair import repair_mesh

# Create a mesh with multiple issues
points = torch.tensor([
    [0.0, 0.0],
    [1.0, 0.0],
    [0.5, 1.0],
    [2.0, 2.0],  # Isolated point
    [0.0, 0.0],  # Duplicate
])

cells = torch.tensor([
    [0, 1, 2],  # Valid
    [0, 0, 1],  # Degenerate
])

mesh = Mesh(points=points, cells=cells)
print(f"Original: {mesh.n_points} points, {mesh.n_cells} cells")

# Repair with detailed stats
repaired, stats = repair_mesh(
    mesh,
    remove_duplicates=True,
    remove_degenerates=True,
    remove_isolated=True,
)

print(f"\nRepaired: {repaired.n_points} points, {repaired.n_cells} cells")
print(f"\nRepair statistics:")
for operation, operation_stats in stats.items():
    print(f"  {operation}: {operation_stats}")

### Individual Repair Operations

In [ ]:
from physicsnemo.mesh.repair.duplicate_removal import remove_duplicate_vertices
from physicsnemo.mesh.repair.degenerate_removal import remove_degenerate_cells
from physicsnemo.mesh.repair.isolated_removal import remove_isolated_vertices

# Example: just remove duplicates
mesh, dup_stats = remove_duplicate_vertices(mesh_with_duplicates, tolerance=1e-6)
print(f"Duplicate removal: {dup_stats}")

## Section 5: Topology Checks

Check if meshes are watertight (closed) or manifold.

In [ ]:
# Closed sphere
sphere = sphere_icosahedral.load(subdivisions=2)
print(f"Sphere:")
print(f"  Watertight: {sphere.is_watertight()}")
print(f"  Manifold: {sphere.is_manifold()}")

In [ ]:
# Hemisphere (open)
hemisphere = sphere.slice_cells(sphere.cell_centroids[:, 2] > 0)
print(f"Hemisphere:")
print(f"  Watertight: {hemisphere.is_watertight()}")
print(f"  Manifold: {hemisphere.is_manifold()}")

In [ ]:
# The bunny (should be watertight if cleaned properly)
bunny = torch.load("assets/bunny.pt", weights_only=False)
print(f"Bunny:")
print(f"  Watertight: {bunny.is_watertight()}")
print(f"  Manifold: {bunny.is_manifold()}")

## Section 6: Practical Workflow

Here's a typical workflow for importing and cleaning external meshes.

In [ ]:
import pyvista as pv
from physicsnemo.mesh.io import from_pyvista

# Load an external mesh
pv_mesh = pv.examples.load_airplane()
mesh = from_pyvista(pv_mesh)

print("Imported mesh:")
print(f"  Points: {mesh.n_points}")
print(f"  Cells: {mesh.n_cells}")

# Step 1: Validate
report = mesh.validate()
print(f"\nValidation: {'PASS' if report['valid'] else 'FAIL'}")

# Step 2: Check topology
print(f"Watertight: {mesh.is_watertight()}")
print(f"Manifold: {mesh.is_manifold()}")

# Step 3: Check quality
quality = mesh.quality_metrics["quality_score"]
print(f"Mean quality: {quality.mean():.3f}")
print(f"Min quality: {quality.min():.3f}")

In [ ]:
# Step 4: Clean if needed
mesh_clean = mesh.clean()

# Step 5: Verify improvements
report_clean = mesh_clean.validate()
print(f"After cleaning:")
print(f"  Points: {mesh_clean.n_points} (was {mesh.n_points})")
print(f"  Cells: {mesh_clean.n_cells} (was {mesh.n_cells})")
print(f"  Valid: {report_clean['valid']}")

## Summary

In this tutorial, you learned about mesh quality and repair:

1. **Quality Metrics**: `mesh.quality_metrics` for per-cell analysis
2. **Statistics**: `mesh.statistics` for mesh summary
3. **Validation**: `mesh.validate()` to detect errors
4. **Repair**:
   - `mesh.clean()` for all-in-one cleaning
   - `repair_mesh()` for detailed control
5. **Topology**: `is_watertight()` and `is_manifold()`

---

### Next Steps

- **Tutorial 6: ML Integration** - Performance benchmarks, datapipes, torch.compile